<a href="https://colab.research.google.com/github/aims-ai-research-foundations/pilot-workshop/blob/main/assignments/day1/day1-course2-student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Long Activity 2 _Lab: Small embeddings and similarity
## Words → Vectors → Geometry

**Student Notebook**

In this lab you'll turn a handful of words into small numeric vectors ("embeddings"), compute distances between them, and ask whether tokens that are *close* in the vector space are also *similar* in meaning. The big idea is **meaning as geometry**.

**Time:** 60 minutes lab work ·

**Linked online activity:** *Lab: Experiment with Embeddings*

### Learning objectives
By the end of this lab you should be able to:
- Define a simple embedding as a numeric vector for each token.
- Compute and interpret Euclidean distances and cosine similarities between embeddings.
- Discuss how the choice of vectors (or training data) shapes which tokens are "close".

### How this notebook is organised
1. **Vocabulary & setup** (provided): run these cells.
2. **Assign embeddings** (provided): random vectors.
3. **Visualise the embedding space** (provided): see the points on a plane.
4. **Implement `euclidean_distance`** (you write it).
5. **Build the distance matrix and find nearest neighbours** (mostly provided).
6. **Implement `cosine_similarity`** (you write it).
7. **Reflect**: do these neighbours match your intuition?


## Step 1: Vocabulary and setup *(just run)*
We start with a tiny vocabulary of 8 tokens.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tokens = [
    "school", "teacher", "student",
    "laptop", "phone",
    "market", "village", "city",
]
print(f'Vocabulary of {len(tokens)} tokens:')
for t in tokens:
    print(' -', t)

## Step 2: Assign random embeddings *(just run)*

We pick `d = 2` (a 2-dimensional vector space easy to visualise) and draw each token's vector from a standard normal.

**Note for later:** these vectors are random, they don't *know* anything about the words. We'll come back to that in the reflection.

In [ ]:
d = 2   # embedding dimension

embeddings = {tok: np.random.randn(d) for tok in tokens}

print('Token embeddings (each row: token -> [dim 1, dim 2])')
for tok, vec in embeddings.items():
    print(f'  {tok:10s} -> [{vec[0]:+.3f}, {vec[1]:+.3f}]')

## Step 3: Visualise the embedding space *(just run)*

Because we chose `d = 2`, every embedding is just a point in a plane. We can plot it. **Look at the picture carefully**, your eyes are doing the same job that the distance function will do later.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for tok, vec in embeddings.items():
    ax.scatter(vec[0], vec[1], s=140,
               color='#04555722', edgecolor='#045557', linewidth=2)
    ax.annotate(tok, (vec[0], vec[1]), xytext=(7, 7),
                textcoords='offset points', fontsize=12, fontweight='bold')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('dim 1'); ax.set_ylabel('dim 2')
ax.set_title('Our token embedding space (2-D)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Reflection**: With your eyes alone, which pairs look closest? Do those pairs match what you'd expect from the *meanings* of the words?

## Step 4: Implement `euclidean_distance` *(you write it)*

The Euclidean distance between two vectors `u = [u1, u2]` and `v = [v1, v2]` is:

$$\text{distance}(u, v) = \sqrt{(u_1 - v_1)^2 + (u_2 - v_2)^2}$$

It generalises to any dimension as $\sqrt{\sum_i (u_i - v_i)^2}$.

### Your task
Complete the function body. Replace the `...` with your implementation.

In [ ]:
def euclidean_distance(u, v):
    """Return the Euclidean distance between two vectors u and v.

    Parameters
    ----------
    u : np.ndarray
        First vector (1-D NumPy array).
    v : np.ndarray
        Second vector (1-D NumPy array, same length as u).

    Returns
    -------
    float
        The Euclidean distance between u and v. Always non-negative.
    """
    # TODO: implement and return the Euclidean distance.
    return ...


# ----- Quick sanity check (don't change) -----
a = np.array([0.0, 0.0])
b = np.array([3.0, 4.0])
got = euclidean_distance(a, b)
print(f'euclidean_distance([0,0], [3,4]) = {got}   (expected: 5.0)')
assert isinstance(got, float), 'Should return a Python float'
assert abs(got - 5.0) < 1e-9, 'Distance from origin to (3,4) is 5.0'
print('Sanity check passed.')

## Step 5: Distance matrix and nearest neighbours *(partly provided)*

Now we use **your** `euclidean_distance` to build a table of distances between every pair of tokens, then find each token's closest neighbour.

### Your task
Fill in **one line** below: call your `euclidean_distance` function on the right pair of embeddings.

In [ ]:
n = len(tokens)
distance_matrix = np.zeros((n, n))

for i, ti in enumerate(tokens):
    for j, tj in enumerate(tokens):
        # TODO: compute the distance between the embeddings of ti and tj,
        #       and store it in distance_matrix[i, j].
        distance_matrix[i, j] = ...

print('Distance matrix (rows and columns are in the order of `tokens`):')
print(np.round(distance_matrix, 3))

Now find each token's nearest other token (smallest *non-zero* distance). **You write the inner search.**

In [ ]:
for i, ti in enumerate(tokens):
    # TODO: among all OTHER tokens (j != i), find the one with the
    #       smallest distance_matrix[i, j].
    # Hint: keep two running variables — best_d (start at infinity) and
    #       best_j (start at None). Loop over j, skip j == i, and update
    #       whenever you find a smaller distance.
    best_j = ...
    best_d = ...
    print(f"Nearest neighbour of '{ti}' is '{tokens[best_j]}' (distance {best_d:.3f})")

**Compare with the scatter plot from Step 3.** Do the printed neighbours match what your eye said? Any surprises?

## Step 6: Implement `cosine_similarity` *(you write it)*

Cosine similarity measures the **angle** between two vectors, ignoring their lengths. The formula is:

$$\cos\text{-sim}(u, v) = \frac{u \cdot v}{\|u\| \, \|v\|}$$

It returns a number between $-1$ and $+1$:
- $+1$ → vectors point in the **same** direction (very similar)
- $0$  → vectors are **perpendicular**
- $-1$ → vectors point in **opposite** directions

### Your task
Complete the function. Keep the docstring.


In [ ]:
def cosine_similarity(u, v):
    """Return the cosine similarity between two vectors u and v.

    Parameters
    ----------
    u : np.ndarray
        First vector (1-D NumPy array).
    v : np.ndarray
        Second vector (1-D NumPy array, same length as u).

    Returns
    -------
    float
        The cosine similarity between u and v, a value in [-1.0, 1.0].
        Returns 0.0 if either vector has zero length.
    """
    # TODO: implement cosine similarity. Handle the zero-vector case.
    return ...


# ----- Quick sanity check (don't change) -----
x = np.array([1.0, 0.0])
y = np.array([1.0, 0.0])
z = np.array([0.0, 1.0])
w = np.array([-1.0, 0.0])
print(f'cos-sim(x, y) = {cosine_similarity(x, y):+.3f}   (expected: +1.000)')
print(f'cos-sim(x, z) = {cosine_similarity(x, z):+.3f}   (expected:  0.000)')
print(f'cos-sim(x, w) = {cosine_similarity(x, w):+.3f}   (expected: -1.000)')
print(f'cos-sim(0, x) = {cosine_similarity(np.zeros(2), x):+.3f}   (expected:  0.000)')

### Apply it to a few interesting pairs

Once your `cosine_similarity` passes the sanity check, use it to compare a few specific token pairs. Pick pairs you *expect* to be similar and pairs you expect to be different.

In [ ]:
# Two pairs are filled in for you. Add at least two more pairs of your own.
pairs = [
    ('school',  'student'),
    ('village', 'city'),
    # TODO: add 2 or 3 more pairs you want to investigate
]

for a, b in pairs:
    sim = cosine_similarity(embeddings[a], embeddings[b])
    print(f"cos-sim({a:10s}, {b:10s}) = {sim:+.3f}")

## Step 7: Reflection

1. **Did the nearest neighbours match your intuition?** Where did they match? Where did they disagree? *(Hint: our vectors were random the embedding space doesn't "know" anything about word meanings.)*
2. If you wanted `school`, `teacher`, `student` to be close together and far from `phone`, `laptop`, what could you do? *(Hint: hand-craft the vectors instead of using random values.)*
3. In real language models, embeddings are **learned from data**. What happens if the training data heavily under-represents certain communities or languages? Whose tokens might end up with poor or unstable embeddings?
4. **Distance vs. cosine similarity:** these are two different ways of measuring closeness. When might one be more useful than the other? *(Hint: think about what happens when one vector is much longer than another.)*


---
### If you finish early
- Replace the random `embeddings` dictionary with **hand-crafted** vectors that encode your intuition (e.g. put `school`, `teacher`, `student` near each other). Rerun the rest of the notebook. Do the nearest neighbours look better now?
- Try `d = 3` and see how the distances change. Can you still visualise?
- Add more tokens from your local context. Where do they land?